# **📘 nb_Gold_DimVendor** #
## **Purpose:** ##
 **Lookup table for network equipment vendors**
### **Logic :** ###

- DISTINCT values

- Generate surrogate key

- Type 1 overwrite

In [1]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

silver_df = spark.table("lh_Silver_Telecom.dbo.silver_network_events")

window_spec = Window.orderBy("vendor_name")

dim_vendor = (
    silver_df
    .select(col("Vendor").alias("vendor_name"))
    .dropna()
    .dropDuplicates()
    .withColumn("vendor_key", row_number().over(window_spec))  
)

(
    dim_vendor
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("lh_Gold_Telecom.dbo.dim_vendor")
)

print("✓ dim_vendor created")



StatementMeta(, bca65262-c110-4d2b-9405-db38e543e90d, 3, Finished, Available, Finished)

✓ dim_vendor created


In [2]:
# --------------------------------------------
# Data Quality Check
# --------------------------------------------

vendor_check = spark.table("lh_Gold_Telecom.dbo.dim_vendor")

print("\n" + "="*50)
print("DIM_VENDOR SUMMARY")
print("="*50)
print(f"Total vendors: {vendor_check.count()}")
print("\nVendor List:")
vendor_check.orderBy("vendor_key").show(truncate=False)
print("="*50)

StatementMeta(, bca65262-c110-4d2b-9405-db38e543e90d, 4, Finished, Available, Finished)


DIM_VENDOR SUMMARY
Total vendors: 3

Vendor List:
+-----------+----------+
|vendor_name|vendor_key|
+-----------+----------+
|VEND_A     |1         |
|VEND_B     |2         |
|VEND_C     |3         |
+-----------+----------+

